# EXCAP Paper Implementation
The code in this notebook is a custom implementation of the EXCAP [paper](https://arxiv.org/abs/2512.01412), "A Self-explainable Model of Long Time Series by Extracting Informative Structured Causal Patterns". This is the companion code of a series of three Medium article's of mine: [Part 1](https://medium.com/@guglielmo.iozzia/implementation-of-a-self-explainable-model-of-long-time-series-by-extracting-informative-structured-e8d96e94d0d5), [Part 2](), and [Part 3](https://medium.com/@guglielmo.iozzia/implementation-of-a-self-explainable-model-of-long-time-series-by-extracting-informative-structured-381ca4199340).  
DISCLAIMER: This isn't an official implementation and I am not tied to the research group that wrote the paper.  
  
While this notebook can run on CPU only, it is highly recommended to execute it on GPU (T4 free tier is enough) to make the model training phase faster.

### Settings

Install the only two missing requirements in the Colab free VMs.

In [ ]:
!pip install sktime torchmetrics

Before running, please ensure your Colab runtime is set to use the T4 GPU: (Runtime -> Change runtime type -> Hardware accelerator: T4 GPU)

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 GPU is available: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("⚠️ GPU not available, using CPU instead.")

### Data Preparation

Download and cache the Wafer dataset.

In [ ]:
from sktime.datasets import load_from_ucr_tsv_to_dataframe, load_UCR_UEA_dataset

X, y = load_UCR_UEA_dataset(name="Wafer")

Prepare the data.

In [ ]:
import numpy as np

# Convert pandas DataFrame where cells are pandas Series to a 3D NumPy array
# EXCAP expects format: [batch_size, channels, sequence_length]
# Since Wafer is univariate, channel dimension is 1.
X_np = np.array([np.array(series) for series in X.iloc[:, 0]])
X_np = X_np.reshape(X_np.shape[0], 1, X_np.shape[1])

# Map labels ('-1.0', '1.0') to standard binary integers (0, 1)
y_unique = np.unique(y)
label_map = {label: i for i, label in enumerate(y_unique)}
y_np = np.array([label_map[label] for label in y], dtype=np.int64)

print("✅ Dataset successfully loaded.")
print(f"   * Raw features (X) shape: {X_np.shape} (Samples, Channels, Timesteps)")
print(f"   * Labels (y) shape: {y_np.shape}")
print(f"   * Class distribution: {np.bincount(y_np)}")

Stratify 80/20 train/test split to maintain class balance.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_np, y_np, test_size=0.2, random_state=42, stratify=y_np
)

print(f"\n✂️ Data Split:")
print(f"   * Train features shape: {X_train.shape}")
print(f"   * Test features shape: {X_test.shape}")

Create a PyTorch DataSet for the Wafer data.

In [ ]:
from torch.utils.data import Dataset

class WaferDataset(Dataset):
    """Custom PyTorch Dataset for loading preprocessed Wafer data."""
    def __init__(self, features, labels):
        # Convert arrays directly to PyTorch tensors for GPU staging
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

Instantiate training and test DataSets.

In [ ]:
train_dataset = WaferDataset(X_train, y_train)
test_dataset = WaferDataset(X_test, y_test)

Create PyTorch DataLoaders.

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("\n📦 PyTorch DataLoaders configured.")
print(f"   * Training Batches: {len(train_loader)}")
print(f"   * Testing Batches: {len(test_loader)}")

print("\n🔍 Verifying DataLoader payload...")
features_batch, labels_batch = next(iter(train_loader))
print(f"   * Features batch shape: {features_batch.shape} (Expected: [{BATCH_SIZE}, 1, 152])")
print(f"   * Labels batch shape: {labels_batch.shape} (Expected: [{BATCH_SIZE}])")
print(f"   * Feature tensor device: {features_batch.device}")

### Model Implementation

Implement the reference model class, a Bi-LSTM plus Attention network, to extract time series temporal attention maps, as for the Section 3.2 of the paper.

In [ ]:
import torch.nn as nn

class ReferenceModel(nn.Module):
    """
    A Bi-LSTM + Attention network used to extract raw
    time series temporal attention maps (Sec 3.2).
    """
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=1):
        super().__init__()
        # Bidirectional LSTM to capture past and future context
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        # Linear layer projecting concatenated Bi-LSTM representations to raw score
        self.attention_proj = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        # Transpose input: [batch, channels, seq_len] -> [batch, seq_len, channels]
        x_trans = x.permute(0, 2, 1)
        lstm_out, _ = self.lstm(x_trans)  # Output shape: [batch, seq_len, hidden_dim * 2]

        # Project hidden states to scalar attention scores
        attention_logits = self.attention_proj(lstm_out).squeeze(-1) # -> [batch, seq_len]
        attention_scores = F.softmax(attention_logits, dim=-1)       # -> [batch, seq_len]
        return attention_scores

Implement the AttentionSegmenter class, which calculates the numerical gradients of the attention maps, finds change-points, and partions the time series into dynamic variable-length segments.

In [ ]:
from scipy.signal import find_peaks

class AttentionSegmenter(nn.Module):
    """
    Calculates numerical gradients of the attention maps, finds change-points,
    and partitions the time series into dynamic variable-length segments (Sec 3.2).
    """
    def __init__(self, reference_model, l_max=16, attention_threshold=0.05):
        super().__init__()
        self.reference_model = reference_model
        self.l_max = l_max
        self.attention_threshold = attention_threshold

    def forward(self, x):
        with torch.no_grad():
            attention_map = self.reference_model(x)

        batch_segments = []
        batch_is_salient = []

        for i in range(x.shape[0]):
            att = attention_map[i]

            att_grad = torch.diff(att, prepend=att.new_zeros(1))

            peaks, _ = find_peaks(att_grad.abs().cpu().numpy(), height=0)

            if len(peaks) > (self.l_max - 1):
                peak_heights = att_grad.abs()[peaks].cpu().numpy()
                top_peak_indices = np.argsort(peak_heights)[-(self.l_max - 1):]
                boundaries = np.sort(peaks[top_peak_indices])
            else:
                boundaries = peaks

            final_boundaries = np.concatenate(([0], boundaries, [x.shape[2]]))
            final_boundaries = np.unique(final_boundaries).astype(int)

            segments = []
            is_salient = []
            for j in range(len(final_boundaries) - 1):
                start, end = final_boundaries[j], final_boundaries[j+1]
                segment = x[i, :, start:end]
                segments.append(segment)

                avg_att = att[start:end].mean()
                is_salient.append(avg_att > self.attention_threshold)

            local_max_len = max(s.shape[1] for s in segments)
            padded_segments = [F.pad(s, (0, local_max_len - s.shape[1])) for s in segments]

            batch_segments.append(torch.stack(padded_segments)) # Shape: [num_segments, channels, local_max_len]
            batch_is_salient.append(torch.tensor(is_salient, dtype=torch.bool, device=x.device))

        return batch_segments, batch_is_salient, attention_map

Implement the Residual 1D Convolutional block.

In [ ]:
class ChainedResidualBlock(nn.Module):
    """Residual 1D convolutional block with dilation and padding."""
    def __init__(self, in_channels, out_channels, kernel_size, stride, dilation, padding):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.relu2 = nn.ReLU()
        self.net = nn.Sequential(self.conv1, self.relu1, self.conv2, self.relu2)

        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None

    def forward(self, x):
        res = x if self.downsample is None else self.downsample(x)
        out = self.net(x)
        return self.relu2(out[:, :, :res.shape[2]] + res)

Implement the TCM Encoder.

In [ ]:
class TCNEncoder(nn.Module):
    """TCN Encoder to map variable-length segments to fixed latent representations."""
    def __init__(self, input_dim=1, num_channels=[32, 64], kernel_size=2):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_ch = input_dim if i == 0 else num_channels[i-1]
            out_ch = num_channels[i]
            padding = (kernel_size - 1) * dilation_size
            layers.append(ChainedResidualBlock(in_ch, out_ch, kernel_size, 1, dilation_size, padding))

        self.tcn = nn.Sequential(*layers)
        self.out_dim = num_channels[-1]

    def forward(self, x):
        feature_maps = self.tcn(x)
        segment_embeddings = feature_maps.mean(dim=2)
        return segment_embeddings

Implement the interpretable Decoder.

In [ ]:
class InterpretableDecoder(nn.Module):
    """
    Dedicated forecasting/classification heads aligned to causal graphs (Sec 3.4).
    Each output dimension is predicted strictly from its structural causal ancestors.
    """
    def __init__(self, embedding_dim, output_dim, n_channels=1):
        super().__init__()
        self.output_dim = output_dim

        self.decoders = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embedding_dim, 64),
                nn.ReLU(),
                nn.Linear(64, 1)
            ) for _ in range(output_dim)
        ])

    def forward(self, segment_embeddings, causal_mask):
        aggregated = segment_embeddings.mean(dim=0, keepdim=True)

        predictions = []
        for d in range(self.output_dim):
            pred = self.decoders[d](aggregated)
            predictions.append(pred)

        return torch.cat(predictions, dim=-1)

Put all things together to build the EXCAP model class.

In [ ]:
class EXCAP(nn.Module):
    """The master EXCAP framework wrapping segmenter, encoder, and causal decoder."""
    def __init__(self, input_dim=1, seq_len=152, n_classes=2, ref_hidden=64, tcn_channels=[32, 64], l_max=16):
        super().__init__()
        self.reference_model = ReferenceModel(input_dim, ref_hidden)
        self.segmenter = AttentionSegmenter(self.reference_model, l_max=l_max)
        self.encoder = TCNEncoder(input_dim, tcn_channels)
        self.decoder = InterpretableDecoder(self.encoder.out_dim, n_classes, input_dim)

    def forward(self, x, causal_mask):
        segments, is_salient, att_map = self.segmenter(x)

        segments_tensor = segments[0].to(x.device)

        embeddings = self.encoder(segments_tensor)

        prediction = self.decoder(embeddings, causal_mask)

        return prediction, embeddings, is_salient[0], att_map

Implement the first of the two latent regularization losses, Separation Loss (L_dist in the paper).

In [ ]:
import torch.nn.functional as F

class SeparationLoss(nn.Module):
    """
    L_dist: Enforces geometric separation (margin-based distance) between
    salient (high-attention) and background (low-attention) latent segment embeddings.
    """
    def __init__(self, margin=2.0):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings, is_salient):
        salient_embeds = embeddings[is_salient]
        background_embeds = embeddings[~is_salient]

        if len(salient_embeds) == 0 or len(background_embeds) == 0:
            return torch.tensor(0.0, device=embeddings.device, requires_grad=True)

        salient_centroid = salient_embeds.mean(dim=0)
        background_centroid = background_embeds.mean(dim=0)

        centroid_distance = torch.norm(salient_centroid - background_centroid, p=2)

        loss = F.relu(self.margin - centroid_distance)
        return loss

Implement the second of the two latent regularization losses, Clustering Loss (L_clus in the paper).

In [ ]:
class ClusteringLoss(nn.Module):
    """
    L_clus: Align segment embeddings to learnable spatial centers (prototypes).
    Encourages semantic clusters in the lower latent dimension.
    """
    def __init__(self, embedding_dim, margin=1.0):
        super().__init__()
        self.margin = margin
        self.salient_prototype = nn.Parameter(torch.randn(1, embedding_dim))
        self.background_prototype = nn.Parameter(torch.randn(1, embedding_dim))
        self.mse_loss = nn.MSELoss()

    def forward(self, embeddings, is_salient):
        salient_embeds = embeddings[is_salient]
        background_embeds = embeddings[~is_salient]

        loss = torch.tensor(0.0, device=embeddings.device, requires_grad=True)

        if len(salient_embeds) > 0:
            loss = loss + self.mse_loss(salient_embeds, self.salient_prototype.expand_as(salient_embeds))
        if len(background_embeds) > 0:
            loss = loss + self.mse_loss(background_embeds, self.background_prototype.expand_as(background_embeds))

        proto_distance = torch.norm(self.salient_prototype - self.background_prototype, p=2)
        loss = loss + F.relu(self.margin - proto_distance)

        return loss

### Training

Define a custom function to handle a training epoch.

In [ ]:
def train_epoch(model, dataloader, optimizer, task_loss_fn, dist_loss_fn, clus_loss_fn, loss_weights, device):
    model.train()
    running_loss = 0.0
    alpha, beta, gamma = loss_weights

    causal_mask = torch.ones(2, 1, device=device)

    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    for features, labels in progress_bar:
        features, labels = features.to(device), labels.to(device)

        for i in range(features.shape[0]):
            optimizer.zero_grad()

            sample_feature = features[i:i+1]
            target_class = labels[i]
            target_one_hot = F.one_hot(target_class, num_classes=2).float()

            pred_logits, seg_embeds, is_salient, _ = model(sample_feature, causal_mask)

            task_loss = task_loss_fn(pred_logits.squeeze(0), target_one_hot)
            dist_loss = dist_loss_fn(seg_embeds, is_salient)
            clus_loss = clus_loss_fn(seg_embeds, is_salient)

            loss = (alpha * task_loss) + (beta * dist_loss) + (gamma * clus_loss)

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        progress_bar.set_postfix({"batch_loss": f" {loss.item():.4f}"})

    return running_loss / len(dataloader.dataset)

Define a custom function to evaluate the model at each epoch.

In [ ]:
from tqdm.auto import tqdm
from torchmetrics.classification import BinaryAUROC, BinaryAveragePrecision

def evaluate_model(model, dataloader, device):
    model.eval()

    auroc_metric = BinaryAUROC().to(device)
    auprc_metric = BinaryAveragePrecision().to(device)

    causal_mask = torch.ones(2, 1, device=device)

    with torch.no_grad():
        for features, labels in tqdm(dataloader, desc="Evaluating", leave=False):
            features, labels = features.to(device), labels.to(device)
            for i in range(features.shape[0]):
                sample_feature = features[i:i+1]

                pred_logits, _, _, _ = model(sample_feature, causal_mask)
                pred_probs = F.softmax(pred_logits.squeeze(0), dim=-1)

                prob_class_1 = pred_probs[1].unsqueeze(0)
                target_class = labels[i].unsqueeze(0)

                auroc_metric.update(prob_class_1, target_class)
                auprc_metric.update(prob_class_1, target_class)

    val_auroc = auroc_metric.compute()
    val_auprc = auprc_metric.compute()
    return val_auroc.item(), val_auprc.item()

Train the EXCAP model (please see inline comments for more details).

In [ ]:
import torch.optim as optim

# Hyperparameters setting
N_EPOCHS = 10
LEARNING_RATE = 1e-3
INPUT_DIM = 1
N_CLASSES = 2

# Instantiate Models, Objectives, and Optimizer
model = EXCAP(input_dim=INPUT_DIM, n_classes=N_CLASSES).to(device)
task_criterion = nn.BCEWithLogitsLoss().to(device)
dist_criterion = SeparationLoss(margin=2.0).to(device)
clus_criterion = ClusteringLoss(embedding_dim=model.encoder.out_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("\n🚀 Commencing EXCAP Model Training Pipeline...")

for epoch in range(N_EPOCHS):
    # Staged optimization (Sec 3.5):
    # Focus heavily on classification accuracy initially (Epochs 0-4),
    # then pivot weights to regularize clustering/explainability (Epochs 5-9).
    if epoch < (N_EPOCHS // 2):
        weights = (1.0, 0.20, 0.05) # Focus on predictive power (alpha, beta, gamma)
    else:
        weights = (1.0, 0.05, 0.20) # Tighten clusters and prototypes

    epoch_loss = train_epoch(
        model, train_loader, optimizer, task_criterion, dist_criterion, clus_criterion, weights, device
    )

    auroc, auprc = evaluate_model(model, test_loader, device)

    print(f"Location: Epoch {epoch+1:02d}/{N_EPOCHS:02d} | "
          f"Loss: {epoch_loss:.4f} | "
          f"AUROC: {auroc:.4f} | "
          f"AUPRC: {auprc:.4f}")

### EXCAP Latent Space Visualization & Interpretation

Define a custom function to extract segment embeddings and metadata.

In [ ]:
def extract_all_segment_embeddings(model, dataloader, device, max_samples=None):
    """
    Passes samples through EXCAP to extract:
      - Latent embeddings z_k in R^{d_z}
      - Saliency flags (True for salient, False for background)
      - Sample class labels (0: Normal, 1: Anomaly)
      - Raw segment waveform slices (for archetype visualization)
    """
    model.eval()

    embeddings_list = []
    saliency_list = []
    labels_list = []
    waveforms_list = []
    scores_list = []

    # Pre-create standard causal mask (2 classes, 1 input channel)
    causal_mask = torch.ones(2, 1, device=device)

    total_samples = 0
    with torch.no_grad():
        for features, labels in tqdm(dataloader, desc="Extracting Segment Latents"):
            features, labels = features.to(device), labels.to(device)
            batch_size = features.shape[0]

            for i in range(batch_size):
                sample_feat = features[i:i+1] # [1, C, T]
                sample_label = labels[i].item()

                # Forward pass through EXCAP
                _, seg_embeds, is_salient, att_map = model(sample_feat, causal_mask)

                # Recompute segment averages to handle threshold fallback if needed
                att = att_map[0]
                segments, _, _ = model.segmenter(sample_feat)
                raw_segments = segments[0]

                # Recalculate segment boundaries
                att_grad = torch.diff(att, prepend=att.new_zeros(1))
                from scipy.signal import find_peaks
                peaks, _ = find_peaks(att_grad.abs().cpu().numpy(), height=0)
                if len(peaks) > (model.segmenter.l_max - 1):
                    peak_heights = att_grad.abs()[peaks].cpu().numpy()
                    top_peak_indices = np.argsort(peak_heights)[-(model.segmenter.l_max - 1):]
                    boundaries = np.sort(peaks[top_peak_indices])
                else:
                    boundaries = peaks
                final_boundaries = np.concatenate(([0], boundaries, [sample_feat.shape[2]]))
                final_boundaries = np.unique(final_boundaries).astype(int)

                for k in range(seg_embeds.shape[0]):
                    embeddings_list.append(seg_embeds[k].cpu().numpy())
                    saliency_list.append(is_salient[k].item())
                    labels_list.append(sample_label)
                    waveforms_list.append(raw_segments[k, 0].cpu().numpy())

                    # Track the mean attention score of this segment
                    start, end = final_boundaries[k], final_boundaries[k+1]
                    scores_list.append(att[start:end].mean().item())

                total_samples += 1
                if max_samples and total_samples >= max_samples:
                    break
            if max_samples and total_samples >= max_samples:
                break

    X_emb = np.array(embeddings_list)
    y_salient = np.array(saliency_list, dtype=bool)
    y_class = np.array(labels_list, dtype=int)
    segment_scores = np.array(scores_list)

    # Fallback: if zero segments are salient, select the top 10% highest attention segments instead
    if np.sum(y_salient) == 0:
        print("\n⚠️ Warning: 0 segments met the saliency threshold.")
        cutoff = np.percentile(segment_scores, 90)
        y_salient = segment_scores >= cutoff
        print(f"  -> Applied fallback: marking the top 10% highest attention segments (score >= {cutoff:.4f}) as salient.")

    print(f"\nExtracted {len(X_emb)} total segments across {total_samples} time-series:")
    print(f"  * Salient segments (High Attention): {np.sum(y_salient)}")
    print(f"  * Background segments (Low Attention): {np.sum(~y_salient)}")

    return X_emb, y_salient, y_class, waveforms_list

Use the function defined in the previous code cell to extract segments from the test set.

In [ ]:
X_emb, y_salient, y_class, raw_waveforms = extract_all_segment_embeddings(
    model, test_loader, device, max_samples=300
)

Do t-SNE dimentionality reduction.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

print("\nComputing 2D t-SNE projection...")
tsne = TSNE(
    n_components=2,
    perplexity=35,
    learning_rate="auto",
    init="pca",
    max_iter=1000,
    random_state=42
)

embeddings_2d = tsne.fit_transform(X_emb)
print("t-SNE projection completed successfully.")

# Compute Silhouette Score on the high-attention subspace to quantify cluster separation
if np.sum(y_salient) > 10:
    salient_2d = embeddings_2d[y_salient]
    salient_classes = y_class[y_salient]
    if len(np.unique(salient_classes)) > 1:
        sil_score = silhouette_score(salient_2d, salient_classes)
        print(f"Salient Subspace Class Silhouette Score: {sil_score:.3f}")

Discover waveform achetypes via K-Means on salient embeddings.

In [ ]:
from sklearn.cluster import KMeans

N_ARCHETYPES = 3 # Number of salient waveform motifs to highlight (Fig 3e style)
salient_indices = np.where(y_salient)[0]

kmeans = KMeans(n_clusters=N_ARCHETYPES, random_state=42, n_init=10)
cluster_ids = kmeans.fit_predict(embeddings_2d[salient_indices])

# Locate the exemplar closest to each cluster center
exemplar_indices = []
for c_id in range(N_ARCHETYPES):
    pts_in_cluster = np.where(cluster_ids == c_id)[0]
    center = kmeans.cluster_centers_[c_id]
    dists = np.linalg.norm(embeddings_2d[salient_indices[pts_in_cluster]] - center, axis=1)
    best_idx = salient_indices[pts_in_cluster[np.argmin(dists)]]
    exemplar_indices.append(best_idx)

Build a visualization with waveform insets, similar to Figure 3(e) in the paper. Using matplotlib here, but you can build it using a more elegant visualization package.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig = plt.figure(figsize=(13, 9), facecolor="white")
ax = fig.add_subplot(111)

# 1. Plot Low-Attention Background (Diffuse Gray Cloud)
bg_mask = ~y_salient
ax.scatter(
    embeddings_2d[bg_mask, 0],
    embeddings_2d[bg_mask, 1],
    c="#CBD5E1",
    s=24,
    alpha=0.45,
    edgecolors="none",
    label="Background Segments (Low Attention)"
)

# 2. Plot Salient Segments by Class
class_0_mask = y_salient & (y_class == 0)
class_1_mask = y_salient & (y_class == 1)

ax.scatter(
    embeddings_2d[class_0_mask, 0],
    embeddings_2d[class_0_mask, 1],
    c="#2563EB", # Blue
    s=45,
    alpha=0.8,
    edgecolors="#1E3A8A",
    linewidths=0.5,
    label="Class 0 Salient (Normal Patterns)"
)

ax.scatter(
    embeddings_2d[class_1_mask, 0],
    embeddings_2d[class_1_mask, 1],
    c="#DC2626", # Red
    s=55,
    alpha=0.85,
    edgecolors="#7F1D1D",
    linewidths=0.5,
    label="Class 1 Salient (Defect / Anomaly Patterns)"
)

# 3. Add Waveform Inset Plots for Key Archetypes
archetype_colors = ["#059669", "#7C3AED", "#D97706"] # Green, Purple, Amber
archetype_titles = ["Archetype A (Transient Shift)", "Archetype B (Oscillatory Pulse)", "Archetype C (Anomaly Surge)"]

# Inset bounding boxes positioned around the main canvas [x, y, width, height] in axes fraction
inset_locs = [
    [0.05, 0.72, 0.22, 0.18],  # Top-left
    [0.72, 0.72, 0.22, 0.18],  # Top-right
    [0.72, 0.10, 0.22, 0.18]   # Bottom-right
]

for idx, (ex_idx, loc, col, title) in enumerate(zip(exemplar_indices, inset_locs, archetype_colors, archetype_titles)):
    ex_pt = embeddings_2d[ex_idx]

    # Highlight centroid point on the main scatter plot
    ax.scatter(ex_pt[0], ex_pt[1], s=160, facecolors="none", edgecolors=col, linewidths=2.5, zorder=5)

    # Draw Inset Axis
    ax_inset = ax.inset_axes(loc)
    wf = raw_waveforms[ex_idx]
    ax_inset.plot(wf, color=col, lw=1.6)
    ax_inset.set_title(title, fontsize=8.5, fontweight="bold", color=col)
    ax_inset.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax_inset.set_facecolor("#F8FAFC")
    for spine in ax_inset.spines.values():
        spine.set_edgecolor(col)
        spine.set_linewidth(1.2)

    # Draw arrow connecting centroid point to inset box
    ax.annotate(
        "",
        xy=(ex_pt[0], ex_pt[1]),
        xytext=(loc[0] + 0.11, loc[1]),
        textcoords=ax.transAxes,
        arrowprops=dict(arrowstyle="->", color=col, lw=1.2, ls="--")
    )

ax.set_title(
    "t-SNE Projection of EXCAP Latent Space $\\mathcal{Z}$_k$\\n(Waveform Archetype Separation vs. Diffuse Background)",
    fontsize=14,
    fontweight="bold",
    pad=15
)
ax.set_xlabel("t-SNE Dimension 1", fontsize=11)
ax.set_ylabel("t-SNE Dimension 2", fontsize=11)
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(loc="lower left", framealpha=0.9, fontsize=9.5)

plt.tight_layout()
plt.show()

### Visualization and Explainability

Define a custom function which plots a given time series sample, highlights the segments identified by the model as salient (highly important), and also plots the raw temporal attention map underneath.

In [ ]:
def visualize_explanation(model, sample_tensor, label, device):
    model.eval()
    # Create a causal mask for the 2 classes and 1 input channel
    causal_mask = torch.ones(2, 1, device=device)
    with torch.no_grad():
        # Force sample tensor to float32 to prevent float64 mismatch with weights
        sample_tensor_f32 = sample_tensor.unsqueeze(0).to(device=device, dtype=torch.float32)
        _, _, is_salient_tensor, att_map = model(sample_tensor_f32, causal_mask)

        # Ensure we convert the saliency indicators into a list of boolean values
        if isinstance(is_salient_tensor, torch.Tensor):
            # If it's a 0-D scalar tensor, wrap it in a list; otherwise convert to list/numpy
            if is_salient_tensor.dim() == 0:
                is_salient = [bool(is_salient_tensor.item())]
            else:
                is_salient = is_salient_tensor.cpu().numpy().tolist()
        else:
            is_salient = list(is_salient_tensor)

        att_map = att_map[0]

        # Re-get boundaries for plotting
        att_grad = torch.diff(att_map, prepend=att_map.new_zeros(1))
        peaks, _ = find_peaks(att_grad.abs().cpu().numpy(), height=1e-4)
        boundaries = np.unique(np.concatenate(([0], peaks, [sample_tensor.shape[1]]))).astype(int)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    ax1.plot(sample_tensor.squeeze().cpu(), color='black', lw=1)
    for i in range(len(boundaries) - 1):
        if i < len(is_salient) and is_salient[i]:
            ax1.axvspan(boundaries[i], boundaries[i+1], color='red', alpha=0.3)
    ax1.set_title(f'Wafer Signal (Label: {"Anomaly" if label == 1 else "Normal"})')
    ax2.plot(att_map.cpu(), color='crimson'); ax2.fill_between(range(len(att_map)), att_map.cpu(), color='crimson', alpha=0.2)
    ax2.set_title("Attention Map")
    plt.show()

Run visualization on one normal and one anomaly sample.

In [ ]:
normal_idx = np.where(y_test == 0)[0][0]
anomaly_idx = np.where(y_test == 1)[0][0]

print("\nExplanation for a Normal Sample:")
visualize_explanation(model, torch.from_numpy(X_test[normal_idx]), y_test[normal_idx], device)

print("\nExplanation for an Anomaly Sample:")
visualize_explanation(model, torch.from_numpy(X_test[anomaly_idx]), y_test[anomaly_idx], device)